# 01 · Exploración de datos — Bayer Leverkusen 2023/24

Primer contacto con StatsBomb Open Data para la temporada del título del Leverkusen
(`competition_id=9`, `season_id=281`). Usamos el loader del paquete, que cachea en
`data/cache/` para no descargar de la red en cada ejecución.

In [ ]:
from pitchiq.data.loader import load_matches, load_events, load_frames

matches = load_matches()
print(f"{len(matches)} partidos")
matches[["match_id", "match_date", "home_team", "away_team", "home_score", "away_score"]].head(10)

In [ ]:
# Eventos de un partido: contiene los DOS equipos
match_id = int(matches.iloc[0]["match_id"])
events = load_events(match_id)
print(events.shape)
events["type"].value_counts().head(15)

In [ ]:
# Acciones defensivas candidatas para M1
defensivas = ["Ball Recovery", "Pressure", "Interception", "Duel"]
events[events["type"].isin(defensivas)].groupby(["team", "type"]).size().unstack(fill_value=0)

In [ ]:
# Datos 360 (freeze-frames): no todos los partidos los tienen
frames = load_frames(match_id)
print(f"{len(frames)} filas de freeze-frame" if not frames.empty else "sin 360 para este partido")
frames.head() if not frames.empty else None

In [ ]:
# Métrica M1: zonas de recuperación + PPDA
from pitchiq.metrics.pressing import recovery_zones, ppda

zones = recovery_zones(events, "Bayer Leverkusen")
print(f"total acciones defensivas: {zones.total}")
print(f"PPDA: {ppda(events, 'Bayer Leverkusen'):.2f}")
zones.counts